In [1]:
import pyarrow
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry


In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

break
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 51.5,
	"longitude": -0.113,
	"start_date": "2012-01-01",
	"end_date": "2026-01-01",
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "rain", "wind_speed_10m", "cloud_cover"],
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["cloud_cover"] = hourly_cloud_cover

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 51.49384689331055°N -0.16302490234375°E
Elevation: 9.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                             date  temperature_2m  relative_humidity_2m  \
0      2012-01-01 00:00:00+00:00       11.191000             94.504494   
1      2012-01-01 01:00:00+00:00       11.391000             95.144356   
2      2012-01-01 02:00:00+00:00       11.490999             95.148079   
3      2012-01-01 03:00:00+00:00       11.591000             94.207436   
4      2012-01-01 04:00:00+00:00       11.591000             93.582436   
...                          ...             ...                   ...   
122755 2026-01-01 19:00:00+00:00        4.200000             77.693863   
122756 2026-01-01 20:00:00+00:00        3.950000             79.927994   
122757 2026-01-01 21:00:00+00:00        3.750000             80.477196   
122758 2026-01-01 22:00:00+00:00        3.600000             80.746681   
122759 2026-01-01 23:00:00+00:00        3.300000             80.1

In [ ]:
weatherdata = pd.read_csv("weatherdata")
weatherdata.to_parquet("weatherdataparq.parquet")

In [ ]:
weatherdata